In [1]:
print("hello from my project")

hello from my project


In [ ]:
import torch
import pandas as pd
import torch.optim as optim

from tqdm import tqdm

from datasets import load_dataset

from torch.utils.data import (
    Dataset,
    DataLoader,
    random_split
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

#load dataset and choose the needed colmuns
dataset = load_dataset("QuyenAnhDE/Diseases_Symptoms")
df = dataset["train"].to_pandas()
df = df[["Name", "Symptoms"]]


Repo card metadata block was not found. Setting CardData to empty.


In [ ]:
df.describe()

,Name,Symptoms
count,400,400
unique,392,395
top,Sciatica,"Swelling, pain, dry mouth, bad taste"
freq,3,3


In [ ]:
# Load Model & Tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
#
tokenizer.pad_token = tokenizer.eos_token # End Of Sequence(use it because gpt2 dont have a padding token to fill the null space)
#
model = AutoModelForCausalLM.from_pretrained(model_name)  # load gpt2 => 124 Million Parameters

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
# type(tokenizer)
# type(model)

transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel

In [ ]:
# # Dataset Class
# # ==========================================

class DiseaseDataset(Dataset):

    def __init__(self, df, tokenizer, max_length=128):

        self.data = df.to_dict(orient="records")
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        disease = self.data[idx]["Name"]
        symptoms = self.data[idx]["Symptoms"]

        text = f"{disease} | {symptoms}"

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

In [ ]:
#  Create Dataset
# ==========================================

full_dataset = DiseaseDataset(df, tokenizer)

train_size = int(0.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size

train_dataset, valid_dataset = random_split(
    full_dataset,
    [train_size, valid_size]
)

batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
# Device
#  ==========================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model.to(device)

print(f"Using device: {device}")

# ==========================================
# Optimizer
# ==========================================

optimizer = optim.AdamW(
    model.parameters(),
    lr=5e-5
)

Using device: cuda


In [ ]:
# ==========================================
# Results DataFrame
# ==========================================

results = pd.DataFrame(
    columns=[
        "epoch",
        "training_loss",
        "validation_loss"
    ]
)

In [ ]:
# training loop
import torch.nn as nn
from tqdm import tqdm

# criterion = nn.CrossEntropyLoss(ignore_index = tokenizer.pad_token_id)

num_epocs = 8

for epoch in range(num_epocs):
    print(f"\n number of epocs {epoch+1}/{num_epocs}")

    model.train()

    train_iterator = tqdm(train_loader, desc=f"Training epoch {epoch+1}")
    epoch_training_loss = 0

    for batch in train_iterator:
        optimizer.zero_grad()

        input_ids = batch["input_ids"]
        if input_ids.dim() == 3 and input_ids.size(1) == 1:
            input_ids = input_ids.squeeze(1)  # same as clone

        input_ids = input_ids.to(device)
        input_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=input_mask, labels=input_ids)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

        train_iterator.set_postfix({
            "Training Loss": loss.item()
        })

    avg_epoch_training_loss = epoch_training_loss / len(train_loader)
    print(f"✅ Average Loss: {avg_epoch_training_loss:.4f}")


 number of epocs 1/8


Training epoch 1: 100%|██████████| 40/40 [00:11<00:00,  3.39it/s, Training Loss=0.0628]


✅ Average Loss: 0.0480

 number of epocs 2/8


Training epoch 2: 100%|██████████| 40/40 [00:12<00:00,  3.31it/s, Training Loss=0.0724]


✅ Average Loss: 0.0418

 number of epocs 3/8


...

✅ Average Loss: 0.0350

 number of epocs 7/8


Training epoch 7: 100%|██████████| 40/40 [00:12<00:00,  3.16it/s, Training Loss=0.0344]


✅ Average Loss: 0.0309

 number of epocs 8/8


Training epoch 8: 100%|██████████| 40/40 [00:11<00:00,  3.42it/s, Training Loss=0.0715]

✅ Average Loss: 0.0297


In [ ]:
import time

# ========== التقييم (Validation) ==========
# إيقاف التعلم
model.eval()
epoch_validation_loss = 0

# شريط التقدم للتقييم
valid_iterator = tqdm(valid_loader, desc=f"Validation Epoch {epoch+1}/{num_epochs}")

with torch.no_grad():
    for batch in valid_iterator:

        input_ids = batch["input_ids"]
        if input_ids.dim() == 3 and input_ids.size(1) == 1:
            input_ids = input_ids.squeeze(1)

        input_ids = input_ids.to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids  #
        )

        loss = outputs.loss

        epoch_validation_loss += loss.item()

        valid_iterator.set_postfix({
            'Validation Loss': f'{loss.item():.4f}'
        })

avg_epoch_validation_loss = epoch_validation_loss / len(valid_loader)

# end_time = time.time()
# epoch_duration_sec = end_time - start_time

# calc outputs
new_row = {
    'transformer': model_name,
    'batch_size': batch_size,
    'gpu': gpu,
    'epoch': epoch + 1,
    'training_loss': avg_epoch_training_loss,
    'validation_loss': avg_epoch_validation_loss,
    # 'epoch_duration_sec': epoch_duration_sec
}

results.loc[len(results)] = new_row

print(f"Epoch: {epoch+1}, Validation Loss: {avg_epoch_validation_loss:.4f}")

# ========== اختبار التوليد (Generation Test) ==========
input_str = "Kidney Failure"
input_ids = tokenizer.encode(input_str, return_tensors='pt').to(device)

output = model.generate(
    input_ids,
    max_length=20,
    num_return_sequences=1,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.1,
    repetition_penalty=1.2
)

decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"\n Generated Text: {decoded_output}")

# ========== save model ==========

torch.save(model, 'SmallMedLM.pt')
print("💾 Model saved locally as 'SmallMedLM.pt'")

#save to google drive
try:
    torch.save(model, 'drive/My Drive/SmallMedLM.pt')
    print("Model saved to Google Drive")
except:
    print("Could not save to Google Drive (Drive not mounted)")

Validation Epoch 8/8: 100%|██████████| 10/10 [00:00<00:00, 10.81it/s, Validation Loss=1.2458]
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Epoch: 8, Validation Loss: 1.0898

 Generated Text: kidney failure [unused489] phosphate [unused925] primaries [unused388] asked [unused282] [unused257]zed [unused10] [unused10] yuki [unused798] [unused10] instead
💾 Model saved locally as 'SmallMedLM.pt'
Could not save to Google Drive (Drive not mounted)
